# Análise do modelo de classificação de coral-sol, primeira e segunda versão, com o dataset PUC-Legado

Autor:  Viviane da Rosa Sommer

Atualizado: 29/12/2024

## Objetivo:
Notebook para fazer a predição das imagens do dataset PUC-Legado pelo modelo V1 e V2 de classificação de coral-sol, para analisar o desempenho do mesmo.

## Importações Necessárias

In [ ]:
!pip install opencv-python-headless
import keras 
import glob 
import numpy as np 
import matplotlib.pyplot as plt 
from keras.preprocessing.image import load_img, img_to_array 
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report 

## Declaração de Constantes e Modelos

In [ ]:
INPUT_DIRECTORY_POSITIVE_IMAGES = "../Coral_Note/puc_dataset/PUC - Coral - Positivo" 
INPUT_DIRECTORY_NEGATIVE_IMAGES = "../Coral_Note/puc_dataset/PUC - Coral - Negativo" 
model_v1 = keras.models.load_model('Pesos/best_weights_v1.keras') 
model_v2 = keras.models.load_model('Pesos/best_weights_v3.keras') 
batch_size = 32 
true_labels = [] 
predicted_labels_v1 = [] 
predicted_labels_v2 = [] 

## Funções Necessárias

In [ ]:
def read_image(filename: str) -> np.ndarray: 
    """ 
    Read an image from a file and convert it to a NumPy array. 
 
    Args: 
        filename (str): The path to the image file. 
 
    Returns: 
        np.ndarray: The cropped image as a NumPy array. 
    """ 
    image = load_img(filename) 
    image = img_to_array(image) 
    height, width, _ = image.shape 
    mid = height // 2 
    top = max(0, mid - int(0.34 * height)) 
    bottom = min(height, mid + int(0.34 * height)) 
    cropped_image = image[top:bottom, :] 
    return cropped_image 
 
def pad_sub_image(sub_image: np.ndarray, size: int) -> np.ndarray: 
    """ 
    Pad a sub-image to ensure it is of the given size. 
 
    Args: 
        sub_image (np.ndarray): The original sub-image. 
        size (int): The desired size of the sub-image. 
 
    Returns: 
        np.ndarray: The padded sub-image. 
    """ 
    padded_image = np.zeros((size, size, 3), dtype=sub_image.dtype) 
    padded_image[:sub_image.shape[0], :sub_image.shape[1], :] = sub_image 
    return padded_image 
 
def divide_image(image: np.ndarray, size: int) -> tuple[list, list]: 
    """ 
    Divide an image into sub-images of a given size. 
 
    Args: 
        image (np.ndarray): The original image. 
        size (int): The size of each sub-image. 
 
    Returns: 
        tuple[list, list]: List of sub-images and list of (x, y) coordinates of the sub-images. 
    """ 
    sub_images = [] 
    coordinates = [] 
    height, width, _ = image.shape 
    for y in range(0, height, size): 
        for x in range(0, width, size): 
            sub_image = image[y:min(y+size, height), x:min(x+size, width)] 
            sub_image = pad_sub_image(sub_image, size) 
            sub_images.append(sub_image) 
            coordinates.append((x, y)) 
    return sub_images, coordinates 
 
def batch_predict(model: keras.Model, data: list, batch_size: int) -> np.ndarray: 
    """ 
    Predict in batches to avoid memory issues. 
 
    Args: 
        model (keras.Model): The trained model. 
        data (list): The data to predict. 
        batch_size (int): The size of each batch. 
 
    Returns: 
        np.ndarray: The predictions. 
    """ 
    predictions = [] 
    for i in range(0, len(data), batch_size): 
        batch = np.array(data[i:i+batch_size]) 
        batch_predictions = model.predict(batch, verbose=0) 
        predictions.extend(batch_predictions) 
    return np.array(predictions) 
 
def plot_results(image: np.ndarray, results_v1: np.ndarray, results_v2: np.ndarray,   
                 coordinates_v1: list, coordinates_v2: list, title: str) -> None: 
    """ 
    Plot the image and mark sub-images as positive or negative for both models. 
 
    Args: 
        image (np.ndarray): The input image. 
        results_v1 (np.ndarray): The results from model V1. 
        results_v2 (np.ndarray): The results from model V2. 
        coordinates_v1 (list): The coordinates for model V1 sub-images. 
        coordinates_v2 (list): The coordinates for model V2 sub-images. 
        title (str): The title for the plot. 
 
    Returns: 
        None 
    """ 
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10)) 
 
    def plot_single_model(ax, image, results, coordinates, size, model_name): 
        ax.imshow(image.astype(np.uint8)) 
        for x, y in coordinates: 
            rect = plt.Rectangle((x, y), size, size, linewidth=1, edgecolor='blue', facecolor='none') 
            ax.add_patch(rect) 
        for idx, (x, y) in enumerate(coordinates): 
            if results[idx]: 
                rect = plt.Rectangle((x, y), size, size, linewidth=2, edgecolor='red', facecolor='none') 
                ax.add_patch(rect) 
        ax.set_title(f"Model {model_name} prediction ({size}x{size})\n{title}") 
 
    plot_single_model(ax1, image, results_v1, coordinates_v1, 128, "V1") 
    plot_single_model(ax2, image, results_v2, coordinates_v2, 128, "V2") 
 
    plt.tight_layout() 
    plt.show() 
 
def process_images(directory: str, true_label: str) -> None: 
    """ 
    Process images in a directory and predict using two models. 
 
    Args: 
        directory (str): The directory containing the images. 
        true_label (str): The true label for the images in the directory. 
 
    Returns: 
        None 
    """ 
    for filename in glob.glob(f"{directory}/**"): 
        try: 
            if filename.split(".")[-1] == "json": 
                continue 
 
            image = read_image(filename) 
            if image is None: 
                continue 
 
            sub_images_v1, coordinates_v1 = divide_image(image, 128) 
            sub_images_v1 = np.array(sub_images_v1) / 255.0 
            results_v1 = batch_predict(model_v1, sub_images_v1, batch_size) 
            binary_results_v1 = results_v1 >= 0.5 
 
            sub_images_v2, coordinates_v2 = divide_image(image, 128) 
            sub_images_v2 = np.array(sub_images_v2) / 255.0 
            results_v2 = batch_predict(model_v2, sub_images_v2, batch_size) 
            binary_results_v2 = results_v2 >= 0.5 
 
            predicted_label_v1 = 'Positive' if np.any(binary_results_v1) else 'Negative' 
            predicted_label_v2 = 'Positive' if np.any(binary_results_v2) else 'Negative' 
 
            true_labels.append(true_label) 
            predicted_labels_v1.append(predicted_label_v1) 
            predicted_labels_v2.append(predicted_label_v2) 
 
            print(f"{filename} = V1: {predicted_label_v1}, V2: {predicted_label_v2}") 
 
            title = f"True label = {true_label}" 
            plot_results(image, binary_results_v1, binary_results_v2, coordinates_v1, coordinates_v2, title) 
        except Exception as e: 
            print(f"Error processing file {filename}: {e}") 
            continue 
 
def plot_confusion_matrices(true_labels: list, predicted_labels_v1: list, predicted_labels_v2: list) -> None: 
    """ 
    Plot confusion matrices for two models and print classification reports. 
 
    Args: 
        true_labels (list): List of true labels. 
        predicted_labels_v1 (list): List of predicted labels from model V1. 
        predicted_labels_v2 (list): List of predicted labels from model V2. 
 
    Returns: 
        None 
    """ 
    true_labels_binary = [1 if label == 'Positive' else 0 for label in true_labels] 
    predicted_labels_binary_v1 = [1 if label == 'Positive' else 0 for label in predicted_labels_v1] 
    predicted_labels_binary_v2 = [1 if label == 'Positive' else 0 for label in predicted_labels_v2] 
 
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8)) 
 
    cm_v1 = confusion_matrix(true_labels_binary, predicted_labels_binary_v1) 
    disp_v1 = ConfusionMatrixDisplay(confusion_matrix=cm_v1, display_labels=['Negative', 'Positive']) 
    disp_v1.plot(ax=ax1, cmap='Blues') 
    ax1.set_title("Confusion Matrix - Model V1") 
 
    cm_v2 = confusion_matrix(true_labels_binary, predicted_labels_binary_v2) 
    disp_v2 = ConfusionMatrixDisplay(confusion_matrix=cm_v2, display_labels=['Negative', 'Positive']) 
    disp_v2.plot(ax=ax2, cmap='Blues') 
    ax2.set_title("Confusion Matrix - Model V2") 
 
    plt.tight_layout() 
    plt.show() 
 
    print("Classification Report - Model V1:") 
    print(classification_report(true_labels_binary, predicted_labels_binary_v1, target_names=['Negative', 'Positive'])) 
    print("\nClassification Report - Model V2:") 
    print(classification_report(true_labels_binary, predicted_labels_binary_v2, target_names=['Negative', 'Positive'])) 

## Processamento das imagens - Casos Positivos

In [ ]:
process_images(INPUT_DIRECTORY_POSITIVE_IMAGES, 'Positive') 

## Processamento das imagens - Casos Negativos

In [ ]:
process_images(INPUT_DIRECTORY_NEGATIVE_IMAGES, 'Negative') 

## Métricas dos resultados

In [ ]:
plot_confusion_matrices(true_labels, predicted_labels_v1, predicted_labels_v2) 

In [ ]:
!jupyter nbconvert --to html analize_model_comparacao.ipynb